<a href="https://www.kaggle.com/code/leonardoterra/shopify-revenue-forecast?scriptVersionId=309648393" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Loading Libraries & Checking Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Libraries for regression models, including OLS
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
from scipy import stats

warnings.filterwarnings('ignore')

In [2]:
dataframe = pd.read_csv('/kaggle/input/datasets/leonardoterra/monthly-revenue-and-spend/shopify_monthly_revenue.csv')
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         26 non-null     object 
 1   period       26 non-null     int64  
 2   seasonality  26 non-null     int64  
 3   year         26 non-null     int64  
 4   quarter      26 non-null     int64  
 5   revenue      26 non-null     float64
 6   spend        26 non-null     float64
 7   promo        26 non-null     int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 1.8+ KB


In [3]:
# For SARIMA model

df_log = dataframe
df_log["revenue"] = pd.to_numeric(df_log["revenue"])
df_log["revenue_log"] = np.log(df_log["revenue"])

In [4]:
# Train & Test split
y = dataframe['revenue']
X = dataframe.drop(['date','revenue'], axis=1)

# Add a constant (intercept) to the model
X = sm.add_constant(X)

# Fit the OLS model
model = sm.OLS(y, X)
results = model.fit()

# Print results
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                revenue   R-squared:                       0.994
Model:                            OLS   Adj. R-squared:                  0.992
Method:                 Least Squares   F-statistic:                     497.7
Date:                Tue, 07 Apr 2026   Prob (F-statistic):           7.67e-20
Time:                        12:16:22   Log-Likelihood:                -332.48
No. Observations:                  26   AIC:                             679.0
Df Residuals:                      19   BIC:                             687.8
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const          -0.3828      1.171     -0.327      

# Preparing Data for ML Models

In [5]:
# Librearies and function to get metrics

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
    max_error
)

def regression_metrics(y_true, y_pred, round_digits=4):
    print("Evaluation Metrics:\n")

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    max_err = max_error(y_true, y_pred)

    print(f"R² Score: {r2:.{round_digits}f}")
    print(f"MAE: {mae:.{round_digits}f}")
    print(f"MAPE(%): {mape * 100:.{round_digits}f}%")
    print(f"Max Error: {max_err:.{round_digits}f}")

In [6]:
# @title

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features and target
y = dataframe['revenue']
X = dataframe.drop(['date','revenue','revenue_log'], axis=1)

# Time-respectingsplit
cutoff = 5

X_train, X_test = X.iloc[:-cutoff], X.iloc[-cutoff:]
y_train, y_test = y.iloc[:-cutoff], y.iloc[-cutoff:]

# Initialize the StandardScaler
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Loading Models & Metrics

In [7]:
# @title
'''
# Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Initialize and train the Linear Regression model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict(X_test_scaled)

# Print metrics
regression_metrics(y_test, y_pred, round_digits=2)
'''

'\n# Linear Regression\n\nfrom sklearn.linear_model import LinearRegression\nfrom sklearn.metrics import mean_squared_error, r2_score\n\n# Initialize and train the Linear Regression model\nmodel = LinearRegression()\nmodel.fit(X_train_scaled, y_train)\n\n# Make predictions\ny_pred = model.predict(X_test_scaled)\n\n# Print metrics\nregression_metrics(y_test, y_pred, round_digits=2)\n'

In [8]:
# @title
'''
pd.options.display.float_format = '{:.2f}'.format
print(pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}))
'''

"\npd.options.display.float_format = '{:.2f}'.format\nprint(pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}))\n"

**XGB Model**

In [9]:
from xgboost import XGBRegressor

# Define features and target
y = dataframe['revenue']
X = dataframe.drop(['date','revenue','revenue_log'], axis=1)

# 3. Train XGBoost
model_xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)
model_xgb.fit(X_train, y_train)

# 4. Predict and evaluate
y_pred = model_xgb.predict(X_test)
regression_metrics(y_test, y_pred, round_digits=2)

Evaluation Metrics:

R² Score: 0.83
MAE: 472485.85
MAPE(%): 25.66%
Max Error: 1000269.37


In [10]:
pd.options.display.float_format = '{:.2f}'.format
print(pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}))

       Actual  Predicted
21  548629.53  592334.06
22 3579222.33 2755953.25
23 3756222.62 2755953.25
24  732658.66  363680.31
25  620029.67  493821.75


**SARIMA**

In [11]:
# @title
!pip install pmdarima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 15.9 MB/s eta 0:00:00


In [12]:
'''

# @title
# Decompose the series
result = seasonal_decompose(df_log['revenue_log'], model='additive', period=12)

# Add components as columns
df_log['trend'] = result.trend
df_log['seasonal'] = result.seasonal
df_log['residual'] = result.resid

# See the result
result.plot()
plt.gcf().set_size_inches(12, 8)
plt.show()

'''

"\n\n# @title\n# Decompose the series\nresult = seasonal_decompose(df_log['revenue_log'], model='additive', period=12)\n\n# Add components as columns\ndf_log['trend'] = result.trend\ndf_log['seasonal'] = result.seasonal\ndf_log['residual'] = result.resid\n\n# See the result\nresult.plot()\nplt.gcf().set_size_inches(12, 8)\nplt.show()\n\n"

In [13]:
'''
# @title
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Autocorrelation check
fig, ax = plt.subplots(2, 1, figsize=(12,8))
plot_acf(df_log['revenue_log'], lags=12, ax=ax[0])
plot_pacf(df_log['revenue_log'], lags=12, ax=ax[1])
plt.show()
'''

"\n# @title\nfrom statsmodels.graphics.tsaplots import plot_acf, plot_pacf\n\n# Autocorrelation check\nfig, ax = plt.subplots(2, 1, figsize=(12,8))\nplot_acf(df_log['revenue_log'], lags=12, ax=ax[0])\nplot_pacf(df_log['revenue_log'], lags=12, ax=ax[1])\nplt.show()\n"

In [14]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(df_log['revenue_log'])
print('ADF Statistic:', result[0])
print('p-value:', result[1])

# p-value < 0.05 → The series **is stationary** (rejects non-stationary).
# p-value ≥ 0.05 → The series **is non-stationary** (you need to fix it).

ADF Statistic: -3.2895947849877
p-value: 0.015350043185463503


In [15]:
'''

# @title
#Auto Arima
from pmdarima import auto_arima

auto_model = auto_arima(
    df_log['revenue_log'],
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    d=None,
    seasonal=True, m=12, # ← monthly seasonality within weekly data
    trace=True,
    information_criterion='aic'
)
print(auto_model.summary())
'''

"\n\n# @title\n#Auto Arima\nfrom pmdarima import auto_arima\n\nauto_model = auto_arima(\n    df_log['revenue_log'],\n    start_p=0, max_p=3,\n    start_q=0, max_q=3,\n    d=None,\n    seasonal=True, m=12, # ← monthly seasonality within weekly data\n    trace=True,\n    information_criterion='aic'\n)\nprint(auto_model.summary())\n"

In [16]:
# @title
from statsmodels.tsa.statespace.sarimax import SARIMAX

# --- Train/test split ---
train = df_log.iloc[:-1]
test = df_log.iloc[-1:]

# SARIMA
sarima_model = SARIMAX(
    train['revenue_log'],
    order=(0, 1, 1),              # (p, d, q)
    seasonal_order=(1, 0, 0, 12), # (P, D, Q, m)
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

# Forecast
sarima_forecast = sarima_fit.forecast(steps=1)

# Compare forecasts vs actuals
results = pd.DataFrame({
    'actual': test['revenue_log'],
    'sarima': sarima_forecast.values
})

mae_sarimax = (results['actual'] - results['sarima']).abs().mean()
print(f"MAE — SARIMAX: {mae_sarimax:.2f}")

# Diagnostics
#fig, axes = plt.subplots(2, 2, figsize=(20, 8))
#sarima_fit.plot_diagnostics(fig=fig)
#plt.suptitle('SARIMA Residual Diagnostics')
#plt.tight_layout()
#plt.show()

                                      SARIMAX Results                                      
Dep. Variable:                         revenue_log   No. Observations:                   25
Model:             SARIMAX(0, 1, 1)x(1, 0, [], 12)   Log Likelihood                  -6.227
Date:                             Tue, 07 Apr 2026   AIC                             18.453
Time:                                     12:16:29   BIC                             19.908
Sample:                                          0   HQIC                            17.915
                                              - 25                                         
Covariance Type:                               opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ma.L1          0.0454      0.383      0.118      0.906      -0.706       0.797
ar.S.L12       0.6984      

In [17]:
'''
# @title
results['actual'] = np.exp(results['actual'])
results['sarima'] = np.exp(results['sarima'])
results
'''

"\n# @title\nresults['actual'] = np.exp(results['actual'])\nresults['sarima'] = np.exp(results['sarima'])\nresults\n"

In [18]:
# Future forecast - SARIMA MODEL

n_months = 1
history = list(df_log['revenue_log'])
future_predictions = []

for i in range(n_months):
    sarima_model = SARIMAX(
      history,
      order=(0, 1, 1),              # (p, d, q)
      seasonal_order=(1, 0, 0, 12), # (P, D, Q, m)
      enforce_stationarity=False,
      enforce_invertibility=False
)
    sarima_fit = sarima_model.fit(disp=False)

    # Forecast 1 step
    sarima_pred = sarima_fit.forecast(steps=1)

# Results table
results_future = pd.DataFrame({
    'Next Week': range(1, n_months + 1),
    'predicted_revenue': np.exp(sarima_pred).round(2)
})

print(results_future.to_string(index=False))

 Next Week  predicted_revenue
         1          592953.87


# Forecasting

In [19]:
# Future prediction -Regression Models

# Variables

period=28
seasonality=4
year=2026
quarter=2
spend=295000
promo=0

next_X = pd.DataFrame([[period, seasonality, year, quarter, spend, promo]], columns=X.columns)
next_X_scaled = scaler.transform(next_X)

# regression_pred = model.predict(next_X_scaled) # Linear regression
xgb_pred = model_xgb.predict(next_X) # XGBoost
sarimax_pred = np.exp(sarima_pred)[0].round(2) # Sarima

In [20]:
y_pred_ensemble = (xgb_pred[0] + sarimax_pred) / 2

# print(f"Linear Regression: {regression_pred[0]:,.2f}")
print(f"XGBoost: {xgb_pred[0]:,.2f}")
print(f"SARIMA: {sarimax_pred:,.2f}")
print(f"Ensemble: {y_pred_ensemble:,.2f}")

XGBoost: 492,180.19
SARIMA: 592,953.87
Ensemble: 542,567.03
